# Pseudobulk Analytics

In [ ]:
import pseudobulk_analytics_utils_04 as pau
import scanpy as sc
import anndata as ad
import decoupler as dc
import gseapy as gp
import os
import numpy as np
import pandas as pd

%matplotlib inline
import matplotlib.pyplot as plt


## 1. Load in file and keep transcripts of interest

### A. Load correct file

<div class="alert alert-block alert-info">
<b> Open h5ad, make sure correct file and confident in cell type annotations
</div>

In [ ]:
#### Read in file 

adata_query_X =ad.read_h5ad("/tscc/lustre/ddn/scratch/aopatel/adata_mtg_Finalized.h5ad")

In [ ]:
#### Make sure UMAP is same one as in 3b

sc.pl.umap(
    adata_query_X,
    color="predictions",
    frameon=False,
    legend_loc="on data",   # puts numbers on clusters
    legend_fontsize=10,
    legend_fontoutline=2
)

In [ ]:
adata_query_X.obs['diagnosis'] = adata_query_X.obs['Consensus clinical diagnosis']
adata_query_X.obs['cps'] = adata_query_X.obs['Continuous Pseudo-progression Score']

In [ ]:
print(adata_query_X.obs[['sex', 'age_numeric', 'pmi', 'diagnosis','RIN']].dtypes)

### B. Filter for important genes and keep high population cell types

In [ ]:
#### Get rid of all mitochondrial and unannotated gene transcripts

## Create mask
mask = ~(adata_query_X.var_names.str.startswith('MT-') | 
         adata_query_X.var_names.str.startswith('ENSG'))

## Filter
adata_query_X = adata_query_X[:, mask].copy()
print(f"Remaining genes: {adata_query_X.n_vars}")

In [ ]:
#### Get rid of cells types that have <3000 cells total

adata_query_X=pau.low_cell_type_remover(adata_query_X,n=3000)

In [ ]:
# Keep Non-Severely Affected cells
# adata_query_X = adata_query_X[adata_query_X.obs["Severely Affected Donor"] == "N"].copy()

# Confirm that the 'Y' category now has 0 counts
# print(adata_query_X.obs["Severely Affected Donor"].value_counts())

## 2. Use "decoupler" for pseudobulking

<div class="alert alert-block alert-info">
<b> Use decoupler to pseudobulk the snRNAseq data. Sum counts by donor. Make sure to use raw counts
</div>

In [ ]:
pdata = dc.pp.pseudobulk(
    adata_query_X, 
    sample_col='individualID', 
    groups_col='predictions', 
    layer='counts',    # Targets your raw data
    mode='sum'        # Sums the counts
)

print(pdata)

<div class="alert alert-block alert-info">
<b> pseudobulking changes meta data types, these must be changed to correct type again!! 
</div>

In [ ]:
print(pdata.obs[['sex', 'age_numeric', 'pmi', 'diagnosis','RIN','cps']].dtypes)

In [ ]:
# 1. Ensure age_numeric and RIN are floats/ints
pdata.obs['age_numeric'] = pd.to_numeric(pdata.obs['age_numeric'], errors='coerce')
pdata.obs['pmi'] = pd.to_numeric(pdata.obs['pmi'], errors='coerce')
pdata.obs['RIN'] = pd.to_numeric(pdata.obs['RIN'], errors='coerce')
pdata.obs['cps'] = pd.to_numeric(pdata.obs['cps'], errors='coerce')

# 2. Keep sex and diagnosis as categories
pdata.obs['sex'] = pdata.obs['sex'].astype('category')
pdata.obs['diagnosis'] = pdata.obs['diagnosis'].astype('category')

# 3. Final Check
print(pdata.obs[['sex', 'age_numeric', 'pmi', 'diagnosis', 'RIN']].dtypes)

In [ ]:
pdata_filtered = {}

for pred in pdata.obs['predictions'].unique():
    adata_sub = pdata[pdata.obs['predictions'] == pred].copy()
    
    # Perform filter 
    dc.pp.filter_by_expr(
        adata=adata_sub,
        group="diagnosis",
        min_count=10,
        min_total_count=15,
        large_n=10,
        min_prop=0.7,
    )
    dc.pp.filter_by_prop(
        adata=adata_sub,
        min_prop=0.1,
        min_smpls=2,
    )
    
    pdata_filtered[pred] = adata_sub


## 3. DESeq2 for DEGs generation

In [ ]:
output_dir = "mtg_pb_DEGs"

pau.DESeq2_pipeline(pdata_filtered, output_dir)

In [ ]:
deg_breakdown=pau.deg_calculator(output_dir)
deg_breakdown

## 4. GSEA

In [ ]:
#### Libraries to use 

#'GO_Biological_Process_2025'
#'MSigDB_Hallmark_2020'
#'ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X'

gp.get_library_name() # To get list of all available libraries 

In [ ]:
pau.gsea_pipeline(
    input_dir="mtg_pb_DEGs", 
    output_dir="mtg_pb_GSEA", 
    library="GO_Biological_Process_2025"
)

## 5. CollecTRI

In [ ]:
results_df = pd.read_csv('/tscc/nfs/home/aopatel/mtg_pb_DEGs/L2-3_IT_deseq2_results.csv', index_col=0)

In [ ]:
data = results_df[["stat"]].T.rename(index={"stat": "disease.vs.normal"})
data

In [ ]:
collectri = dc.op.collectri(organism="human")
collectri

In [ ]:
# Run
tf_acts, tf_padj = dc.mt.ulm(data=data, net=collectri)

# Filter by sign padj
msk = (tf_padj.T < 0.05).iloc[:, 0]
tf_acts = tf_acts.loc[:, msk]

tf_acts

In [ ]:
dc.pl.barplot(data=tf_acts, name="disease.vs.normal", figsize=(7, 6))

In [ ]:
tf_acts

In [ ]:
dc.pl.network(
    net=collectri,
    data=data,
    score=tf_acts,
    sources=["SP1", "MYC", "TP53", "ESR1","AR"],
    targets=5,
    figsize=(5, 5),
    vcenter=True,
    by_abs=True,
    size_node=15,
)

## 6. PROGENy

In [ ]:
progeny = dc.op.progeny(organism="human")
progeny

In [ ]:
# Run
pw_acts, pw_padj = dc.mt.ulm(data=data, net=progeny)

# Filter by sign padj
msk = (pw_padj.T < 0.05).iloc[:, 0]
pw_acts = pw_acts.loc[:, msk]

pw_acts

In [ ]:
dc.pl.barplot(data=pw_acts, name="disease.vs.normal", figsize=(7, 6))


In [ ]:
# 1. Get all TGFb target genes from PROGENy
tgfb_targets = progeny[progeny['source'] == 'MAPK']

# 2. Find which of those genes are actually in your 'data' columns
# (Using a list comprehension or .isin())
available_genes = [g for g in tgfb_targets['target'] if g in data.columns]

print(f"Found {len(available_genes)} out of {len(tgfb_targets)} TGFb target genes in your data.")

# 3. Filter the weights and the data using only available genes
tgfb_weights_filtered = tgfb_targets.set_index('target').loc[available_genes]
tgfb_expression = data[available_genes]

# 4. Calculate contribution (Expression * Weight)
# This shows you the actual impact of each gene on your final score
contributions = tgfb_expression * tgfb_weights_filtered['weight']

In [ ]:
# Look at the first sample (row 0)
sample_idx = 0
sample_contributions = contributions.iloc[sample_idx].sort_values(ascending=False)

print("Top genes DRIVING TGFb activity in this sample:")
print(sample_contributions.head(10))

print("\nTop genes INHIBITING TGFb activity in this sample:")
print(sample_contributions.tail(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Combine top and bottom for plotting
plot_data = pd.concat([sample_contributions.head(10), sample_contributions.tail(10)])
plot_df = plot_data.reset_index()
plot_df.columns = ['Gene', 'Contribution']

#

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

# 1. Define the normalization centered at 0
# vmin and vmax should be the boundaries of your data
norm = mcolors.TwoSlopeNorm(
    vmin=plot_df['Contribution'].min(), 
    vcenter=0, 
    vmax=plot_df['Contribution'].max()
)

# 2. Map the 'Contribution' values to the 'RdBu_r' colormap
cmap = plt.get_cmap('RdBu_r')
colors = [cmap(norm(value)) for value in plot_df['Contribution']]

# 3. Create the plot
plt.figure(figsize=(8, 8))
sns.barplot(
    data=plot_df, 
    x='Contribution', 
    y='Gene', 
    palette=colors  # Pass the list of gradient colors
)

# Add a vertical line at 0 for context
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.title('MAPK Pathway Drivers')
plt.xlabel('Contribution (Weight * Expression)')
plt.ylabel('Top/Bottom Target Genes')

plt.tight_layout()
plt.show()

In [ ]:
# Check if all gene names are unique
print(f"Are all genes unique? {pbulk.var_names.is_unique}")

# If False, find out which genes are duplicated
duplicate_genes = pbulk.var_names[pbulk.var_names.duplicated()].unique()
print(f"Duplicated genes: {list(duplicate_genes)}")

In [ ]:
print(pd.crosstab(adata_query_X.obs['individualID'], adata_query_X.obs['Consensus clinical diagnosis']))